In [6]:
# Self-supervised pretraining using SimCLR (via lightly)

import torch
from lightly.data import LightlyDataset
from lightly.transforms import SimCLRTransform
from lightly.models import SimCLR
from torchvision.models import resnet18
from lightly.loss import NTXentLoss
from lightly.data.collate import BaseCollateFunction
from torch.utils.data import DataLoader
from torchvision import transforms
import os
from PIL import Image

# === Config ===
data_dir = 'some_images'  # Folder with unlabeled images
batch_size = 3
epochs = 2
learning_rate = 1e-3
checkpoint_path = 'pretrained_streamflow_ai.pth'

# === Transforms ===
# bottom crop: removes 150 pixels from the bottom
def crop_bottom(img: Image.Image, pixels: int = 150):
    return img.crop((0, 0, img.width, img.height - pixels))

class BottomCropTransform:
    def __init__(self, pixels=150):
        self.pixels = pixels

    def __call__(self, img: Image.Image):
        return crop_bottom(img, self.pixels)

simclr_view = SimCLRTransform(input_size=224)
crop_transform = BottomCropTransform(pixels=150)

def custom_collate_fn(batch):
    x0, x1 = [], []
    for img, _, _ in batch:
        cropped = crop_transform(img)
        view1, view2 = simclr_view(cropped)
        x0.append(view1)
        x1.append(view2)
    return torch.stack(x0), torch.stack(x1)

# === Dataset & Dataloader ===
dataset = LightlyDataset(input_dir=data_dir)
dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=custom_collate_fn
)

# === Model, Loss, Optimizer ===
backbone = resnet18(pretrained=False)
backbone.fc = torch.nn.Identity()
model = SimCLR(backbone, num_ftrs=512)
loss_fn = NTXentLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# === Training loop ===
for epoch in range(epochs):
    total_loss = 0
    model.train()
    for x0, x1 in dataloader:
        x0, x1 = x0.to(device), x1.to(device)
        z0, z1 = model(x0), model(x1)
        loss = loss_fn(z0, z1)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {total_loss/len(dataloader):.4f}")

# === Save encoder weights ===
torch.save(model.backbone.state_dict(), checkpoint_path)
print(f"Saved pretrained encoder to {checkpoint_path}")


Epoch [1/2] - Loss: 1.7110
Epoch [2/2] - Loss: 1.5758
Saved pretrained encoder to pretrained_streamflow_ai.pth
